# Blocks 1–5 — Colab pipeline (live tree)

Opening a notebook from GitHub **does not** include `src/`. **Runtime → Run all.** The first code cell must print `BOOTSTRAP_V3` and `med_doc: .../src/med_doc/__init__.py`. If you still see `No module named med_doc`, you are on an old notebook copy: File → Open notebook from GitHub → `RwaRwa599/epq3` branch **`block1`** → `Pipeline_Blocks_1_to_5.ipynb`, then **Runtime → Disconnect and delete runtime**.

Older `block1/`, `block2/`, `block3/` folders are snapshots. Prefer this notebook + `notebooks/Block_*.ipynb`.

**Do not upload clinic PHI to Colab.**


Per-block notebooks: [1](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_1_Document_Normalization.ipynb) · [2](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_2_Knowledge_Graph.ipynb) · [3](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_3_Marks_and_HTR.ipynb) · [4](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_4_KG_Rescoring.ipynb) · [5](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_5_Review_and_LIS.ipynb)


## 0. Clone


In [ ]:
"""Colab/kernel bootstrap for live src/med_doc.

Opening a GitHub notebook does not clone the repo. Run this as the first cell.
Prints BOOTSTRAP_V3 when import med_doc succeeds.
"""

from __future__ import annotations

import os
import site
import subprocess
import sys
from pathlib import Path

BOOTSTRAP_VERSION = "BOOTSTRAP_V3"
REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"


def _run(cmd: list[str]) -> None:
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def _token() -> str | None:
    tok = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


def _clone_url() -> str:
    tok = _token()
    if tok:
        return f"https://{tok}@github.com/RwaRwa599/epq3.git"
    return REPO


def _find_root() -> Path | None:
    here = Path.cwd().resolve()
    for cand in (
        here,
        here.parent,
        Path("/content/epq3"),
        Path("/content") / "epq3",
    ):
        if (cand / "src" / "med_doc" / "__init__.py").is_file():
            return cand
    return None


def _write_pth(src: Path) -> None:
    line = str(src.resolve()) + "\n"
    dirs = []
    try:
        dirs.extend(site.getsitepackages())
    except Exception:
        pass
    try:
        dirs.append(site.getusersitepackages())
    except Exception:
        pass
    sp = Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    dirs.append(str(sp))
    for d in dirs:
        if not d:
            continue
        target = Path(d)
        try:
            target.mkdir(parents=True, exist_ok=True)
            (target / "epq3_src.pth").write_text(line, encoding="utf-8")
            print("wrote", target / "epq3_src.pth")
        except Exception as exc:
            print("pth skip", target, exc)


def _ipython_cd(path: Path) -> None:
    try:
        ip = get_ipython()  # type: ignore[name-defined]
    except Exception:
        ip = None
    if ip is None:
        os.chdir(path)
        return
    ip.run_line_magic("cd", str(path))


def _ipython_pip(root: Path) -> None:
    try:
        ip = get_ipython()  # type: ignore[name-defined]
    except Exception:
        ip = None
    if ip is not None:
        ip.run_line_magic("pip", "install -q matplotlib opencv-python-headless pydantic Pillow numpy")
        ip.run_line_magic("pip", f"install -q -e {root}")
        return
    _run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "opencv-python-headless", "pydantic", "Pillow", "numpy"])
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)])


def put_src_on_path(root: Path | None = None) -> Path:
    root = root or _find_root()
    if root is None:
        raise ModuleNotFoundError(
            "med_doc not found. Run the first notebook cell (BOOTSTRAP_V3 clone). "
            "Private repo: Colab secret GITHUB_TOKEN. Then Runtime → Run all."
        )
    src = (root / "src").resolve()
    os.chdir(root)
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))
    os.environ["PYTHONPATH"] = str(src) + os.pathsep + os.environ.get("PYTHONPATH", "")
    return root


def bootstrap() -> Path:
    print(BOOTSTRAP_VERSION)
    dest = Path("/content/epq3") if Path("/content").is_dir() else (Path.cwd().resolve() / "epq3")
    root = _find_root()
    if root is None:
        url = _clone_url()
        if dest.exists() and not (dest / "src" / "med_doc" / "__init__.py").is_file():
            import shutil

            shutil.rmtree(dest, ignore_errors=True)
        if not (dest / ".git").is_dir():
            _run(["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", url, str(dest)])
        else:
            _run(["git", "-C", str(dest), "fetch", "origin", BRANCH])
            _run(["git", "-C", str(dest), "checkout", BRANCH])
            _run(["git", "-C", str(dest), "pull", "--ff-only", "origin", BRANCH])
        root = dest
    _ipython_cd(root)
    os.chdir(root)
    src = root / "src"
    if str(src.resolve()) not in sys.path:
        sys.path.insert(0, str(src.resolve()))
    _write_pth(src)
    try:
        _ipython_pip(root)
    except Exception as exc:
        print("pip note:", exc)
    # Drop a copy next to cwd as last resort (some Colab kernels ignore .pth until restart)
    try:
        import med_doc  # noqa: F401
    except ModuleNotFoundError:
        sys.path.insert(0, str(src.resolve()))
        import importlib

        importlib.invalidate_caches()
        import med_doc  # noqa: F401
    import med_doc

    print("cwd:", os.getcwd())
    print("med_doc:", med_doc.__file__)
    if "src" not in Path(med_doc.__file__).parts:
        print("warning: unexpected med_doc location")
    return root


root = bootstrap()


Private repo: Colab secret `GITHUB_TOKEN` is read automatically by the clone cell when present.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet

## 1. Choose a batch of images

Same input shapes as Block 1a: a **folder**, a **ZIP of photos**, a **list of paths**, or Colab multi-upload.

Do **not** upload clinic PHI. Default: two copies of the synthetic blank.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

import shutil
from med_doc.pipeline import run_blocks_1_to_5

USE_UPLOAD = False
BATCH_DIR = None  # e.g. Path("/content/photos") or Path("photos.zip")

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded.keys())
    if len(names) == 1 and names[0].lower().endswith(".zip"):
        batch_input = names[0]
    else:
        batch_input = names
elif BATCH_DIR is not None:
    batch_input = BATCH_DIR
else:
    sheet = demo_sheet()
    raw = OUT / "raw_batch"
    raw.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sheet, raw / "form_a.png")
    shutil.copy2(sheet, raw / "form_b.png")
    batch_input = raw

print("batch_input:", batch_input)


## 2. Run Blocks 1–5 on the whole batch

`run_blocks_1_to_5` = `normalize_batch` → Block 3 drafts → Block 4 KG → Block 5 orders.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()

from med_doc.pipeline import run_blocks_1_to_5

pipe = run_blocks_1_to_5(
    batch_input,
    output_dir=OUT,
    backend="lexicon",
    patch_missing_edta=True,  # demo nurse patch when EDTA crop is empty
)
print("docs", pipe["block5"]["manifest"]["total_documents"])
for row in pipe["block5"]["manifest"]["documents"]:
    print(row["doc_id"], "ticked", row.get("n_ticked"), "needs_review", row.get("needs_review"),
          "observed", row.get("observed_tubes"))


## 3. Inspect first document

In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()


import json

docs = pipe["block5"]["manifest"]["documents"]
doc_id = docs[0]["doc_id"]
show_rgb(OUT / "b1" / "docs" / doc_id / "overlay.png", f"Block 1 overlay — {doc_id}", figsize=(12, 10))
hyp = json.loads((OUT / "b3" / "docs" / doc_id / "hypotheses.json").read_text())
pred = json.loads((OUT / "b4" / "docs" / doc_id / "prediction.json").read_text())
order = json.loads((OUT / "b5" / "docs" / doc_id / "order.json").read_text())
print("marked", [k for k, v in hyp["nonverbal"].items() if v["is_marked"]])
print("tube source", hyp["verbal"].get("tube_edta", {}).get("source"))
print("B4 observed", pred.get("observed_tubes"), "expected", pred.get("expected_tubes"))
print("B5 order needs_review", order["needs_review"], "observed", order["observed_tubes"])
assert hyp["verbal"].get("tube_edta", {}).get("source") != "prior_expected"


## 4. Download batch ZIPs

In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    for cand in (Path("/content/epq3"), Path.cwd(), Path.cwd().parent):
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    raise ModuleNotFoundError(
        "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V3 "
        "and med_doc: .../src/med_doc/__init__.py  (Runtime → Run all). "
        "Private repo: Colab secret GITHUB_TOKEN."
    )

_guard()


for name in ("block1.zip", "block3.zip", "block4.zip", "block5.zip"):
    download(OUT / name)
